# LeetCode #1000: Minimum Cost to Merge Stones

https://leetcode.com/problems/minimum-cost-to-merge-stones/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(k^n)$ | $O(n)$ |
| **Optimal: Interval DP with Prefix Sums ★** | $O(n^3 / k)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Recursively simulate all possible merge sequences without memoization. The exponential branching factor makes this infeasible for large inputs.

### Optimal: Interval DP with Prefix Sums ★
`dp[i][j]` stores the minimum cost to reduce `stones[i..j]` to the fewest possible piles. A merge is only possible if `(n-1) % (k-1) == 0`; otherwise return $-1$. Prefix sums give $O(1)$ range-sum queries. The key recurrence: split `[i,j]` at every step of $k-1$, then add the range sum when `(j-i) % (k-1) == 0` (meaning this range can be merged into one pile).

**Constraints:**
* $1 \leq \text{stones.length} \leq 30$
* $1 \leq k \leq 30$
* $1 \leq \text{stones}[i] \leq 100$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MergeStones(int[] stones, int k) {
        int n = stones.Length;
        // A valid merge reduces n piles to 1 only when (n-1) % (k-1) == 0
        if ((n - 1) % (k - 1) != 0) return -1;
        // Prefix sums allow O(1) range-sum queries during DP
        int[] pre = new int[n + 1];
        for (int i = 0; i < n; i++) pre[i + 1] = pre[i] + stones[i];
        int[,] dp = new int[n, n];
        for (int len = k; len <= n; len++) {
            for (int i = 0; i <= n - len; i++) {
                int j = i + len - 1;
                dp[i, j] = int.MaxValue;
                // Split at every k-1 step so each left part ends reducible to 1 pile
                for (int m = i; m < j; m += k - 1)
                    if (dp[i, m] != int.MaxValue && dp[m + 1, j] != int.MaxValue)
                        dp[i, j] = Math.Min(dp[i, j], dp[i, m] + dp[m + 1, j]);
                // When the range length allows it, add the cost of a final merge
                if ((j - i) % (k - 1) == 0)
                    dp[i, j] += pre[j + 1] - pre[i];
            }
        }
        return dp[0, n - 1];
    }
}

### Python

In [ ]:
class Solution:
    def merge_stones(self, stones: list[int], k: int) -> int:
        n = len(stones)
        # A valid merge reduces n piles to 1 only when (n-1) % (k-1) == 0
        if (n - 1) % (k - 1) != 0: return -1
        # Prefix sums allow O(1) range-sum queries during DP
        pre = [0] * (n + 1)
        for i in range(n): pre[i + 1] = pre[i] + stones[i]
        dp = [[0] * n for _ in range(n)]
        for length in range(k, n + 1):
            for i in range(n - length + 1):
                j = i + length - 1
                dp[i][j] = float('inf')
                # Split at every k-1 step so each left part ends reducible to 1 pile
                for m in range(i, j, k - 1):
                    dp[i][j] = min(dp[i][j], dp[i][m] + dp[m + 1][j])
                # When the range length allows it, add the cost of a final merge
                if (j - i) % (k - 1) == 0:
                    dp[i][j] += pre[j + 1] - pre[i]
        return dp[0][n - 1]

### Go

In [ ]:
func mergeStones(stones []int, k int) int {
    n := len(stones)
    // A valid merge reduces n piles to 1 only when (n-1) % (k-1) == 0
    if (n-1)%(k-1) != 0 { return -1 }
    // Prefix sums allow O(1) range-sum queries during DP
    pre := make([]int, n+1)
    for i, s := range stones { pre[i+1] = pre[i] + s }
    dp := make([][]int, n)
    for i := range dp { dp[i] = make([]int, n) }
    for length := k; length <= n; length++ {
        for i := 0; i <= n-length; i++ {
            j := i + length - 1
            dp[i][j] = 1<<31 - 1
            // Split at every k-1 step so each left part ends reducible to 1 pile
            for m := i; m < j; m += k - 1 {
                if dp[i][m] != 1<<31-1 && dp[m+1][j] != 1<<31-1 {
                    if v := dp[i][m] + dp[m+1][j]; v < dp[i][j] { dp[i][j] = v }
                }
            }
            // When the range length allows it, add the cost of a final merge
            if (j-i)%(k-1) == 0 { dp[i][j] += pre[j+1] - pre[i] }
        }
    }
    return dp[0][n-1]
}

### Rust

In [ ]:
impl Solution {
    pub fn merge_stones(stones: Vec<i32>, k: i32) -> i32 {
        let (n, k) = (stones.len(), k as usize);
        // A valid merge reduces n piles to 1 only when (n-1) % (k-1) == 0
        if (n - 1) % (k - 1) != 0 { return -1; }
        // Prefix sums allow O(1) range-sum queries during DP
        let mut pre = vec![0i32; n + 1];
        for i in 0..n { pre[i+1] = pre[i] + stones[i]; }
        let mut dp = vec![vec![0i32; n]; n];
        for length in k..=n {
            for i in 0..=(n - length) {
                let j = i + length - 1;
                dp[i][j] = i32::MAX;
                // Split at every k-1 step so each left part ends reducible to 1 pile
                let mut m = i;
                while m < j {
                    if dp[i][m] != i32::MAX && dp[m+1][j] != i32::MAX {
                        let v = dp[i][m] + dp[m+1][j];
                        if v < dp[i][j] { dp[i][j] = v; }
                    }
                    m += k - 1;
                }
                // When the range length allows it, add the cost of a final merge
                if (j - i) % (k - 1) == 0 { dp[i][j] += pre[j+1] - pre[i]; }
            }
        }
        dp[0][n-1]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `stones = [3, 2, 4, 1]`, `k = 2`
$(n-1)\%(k-1) = 3\%1 = 0$ — valid. The DP finds the optimal merge order: merge $[2,4] \to 6$, then $[3,6] \to 9$, then $[9,1] \to 10$. Total $= 6+9+10 = 25$. A different order costs more.

### 2. Slightly Complex
**Input:** `stones = [3, 5, 1, 2, 6]`, `k = 3`
$(n-1)\%(k-1) = 4\%2 = 0$ — valid. The DP considers all splits in steps of 2. The minimum cost is $25$.

### 3. Edge Case: Time Factor
**Input:** `stones = [1] * 30`, `k = 2`
$n = 30$, $k = 2$: the DP fills $30 \times 30 = 900$ cells, each scanning up to $n/2$ split points. Maximum computation: $O(n^3) = 27{,}000$ operations with all uniform values.

### 4. Edge Case: Space Factor
**Input:** `stones = [5]`, `k = 3`
Single pile, nothing to merge. `dp[0][0] = 0`. Returns $0$ immediately after checking the feasibility condition and filling the trivial base case.

### 5. Almost-Impossible but Plausible
**Input:** `stones = [1, 2, 3]`, `k = 3`
$(3-1)\%(3-1) = 2\%2 = 0$ — valid. Only one merge possible: combine all three at once for cost $1+2+3 = 6$. The DP has exactly one non-trivial cell $(0,2)$, confirming the formula reduces to a single $k$-way merge.